# convT-as-flipped-padded-conv — worked example 1: Rebuild stride-1 ConvTranspose2d as flipped padded Conv2d (multi-channel)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-as-flipped-padded-conv`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A stride-1, no-padding `F.conv_transpose2d(x, weight)` is exactly equal to a regular `F.conv2d` on an input zero-padded by `K-1` on every spatial side, using the kernel flipped along both spatial axes and with its channel axes swapped. ConvT weight layout is `(IC, OC, K, K)`; conv2d wants `(OC, IC, K, K)`, hence the `.transpose(0, 1)`. The spatial flip is what turns the *adjoint* (transpose) operation back into an ordinary correlation.

## Worked solution

We are given `x: (B, IC, H, W)` and a ConvT weight `weight: (IC, OC, K, K)` and we want to reproduce `F.conv_transpose2d(x, weight)` (stride 1, no padding) using only `F.conv2d`.

**Step 1 - pad the input by `K-1`.** Transposed convolution *grows* the spatial dims to `H+K-1`. A regular conv2d (valid padding) *shrinks* by `K-1`. To make conv2d produce the larger output, we first pad `x` with `K-1` zeros on every side: `F.pad(x, (K-1, K-1, K-1, K-1))`. After padding the input is `(B, IC, H+2(K-1), W+2(K-1))`, and valid conv2d will then output `(H+K-1)` along each spatial dim - exactly the ConvT size.

**Step 2 - flip the kernel spatially.** ConvT is the adjoint of conv. The adjoint reverses the kernel taps along each spatial axis, so we apply `weight.flip(-1).flip(-2)`. This is the single most important transform: without it the numbers are simply wrong, even though the shape matches.

**Step 3 - swap the channel axes.** ConvT stores weights as `(IC, OC, K, K)` because it maps `IC -> OC` in the transpose direction. Regular conv2d expects `(OC, IC, K, K)`. So we `.transpose(0, 1)` to reinterpret the same data with the channel roles swapped.

**Step 4 - run conv2d.** `F.conv2d(x_pad, w_flipped_swapped)` now produces a tensor identical to the ConvT output. We confirm with `torch.allclose`.

Why it works: convolution and transposed convolution are adjoint linear maps. The adjoint of a valid (shrinking) correlation with kernel `w` is a full (growing) correlation with the flipped kernel - which is exactly 'pad by K-1, flip, swap channels'.

In [ ]:
import torch.nn.functional as F

def convT_as_padded_conv(x, weight):
    K = weight.shape[-1]
    x_pad = F.pad(x, (K - 1, K - 1, K - 1, K - 1))   # (B, IC, H+2(K-1), W+2(K-1))
    w = weight.flip(-1).flip(-2).transpose(0, 1)     # spatial flip + (IC,OC)->(OC,IC)
    return F.conv2d(x_pad, w)

t.manual_seed(0)
x = t.randn(2, 3, 5, 5)          # (B=2, IC=3, H=5, W=5)
weight = t.randn(3, 4, 3, 3)     # ConvT layout (IC=3, OC=4, K=3)

rebuilt = convT_as_padded_conv(x, weight)
reference = F.conv_transpose2d(x, weight)
print('rebuilt shape:', tuple(rebuilt.shape))
print('matches ConvT:', t.allclose(rebuilt, reference, atol=1e-5))
print('max abs diff:', (rebuilt - reference).abs().max().item())